# DEEZER API PULL

11/5/26 script to pull deezer data from their public API.
- have genre
- can pull top 50 artists in each genre (url format: base + /genre + /genre_id + /artists)
- goal: dataframe of top 50 artists per genre

functions:
- get genre
- get artists from genre
- validation
- make dataframe

ok so the top 50 doesnt work well, what we're gonna do is get the top x tracks across all genres, plays, and get the top artist by plays of each days, top x in most genres, yesterdays, past week, past month (when available). lets also make this just deezer

In [ ]:
import argparse
import requests
import pandas as pd
import time
import boto3

In [45]:
# error class so we know where the error comes from further down the line/when the script is run
class DeezerError(Exception):
    pass

In [88]:
url = 'https://api.deezer.com'

session = requests.Session()
response = session.get(f'{url}/genre/0/artists', timeout=15)

In [46]:
# initial testing
try:
    # response = session.get(f'{url}/genre', timeout=15)
    response.raise_for_status()
    data_json = response.json()
    # print('success')
except requests.RequestException as e: # network, timeout, bad status
    print("request failed:", e)
except DeezerError as e:
    print('failed', e)


# DEFINE FUNCTIONS

In [48]:
def get_genre_data() -> pd.DataFrame:
    try:
        response = session.get(f'{url}/genre', timeout=15)
        response.raise_for_status()
        body = response.json()               # the whole response
        if "error" in body:
            raise DeezerError('error in genres data body')
        df = pd.DataFrame(body["data"])[['id', 'name']]
        return df[df['id'] != 0] # no 'all' genre
    except requests.RequestException as e:
        print("request failed:", e)
        raise
    except DeezerError as e:
        print("deezer error:", e)
        raise

genres = get_genre_data()
genres = genres

In [50]:
def get_top_tracks() -> pd.DataFrame:
    try:
        genres = get_genre_data()
        result = []
        for genre_id, genre in zip(genres['id'], genres['name']):
            response = session.get(f'{url}/chart/{genre_id}/tracks', timeout=15)
            response.raise_for_status()
            body = response.json() # the whole response
            if "error" in body:
                raise DeezerError('error in artists data body')
            # dataframe transformation
            df = pd.DataFrame(body['data'])
            df['Genre'] = genre
            result.append(df)
            # print(len(result))
            # time.sleep(.5)
        return pd.concat(result, ignore_index=True)
    except requests.RequestException as e:
            print("request failed:", e)
            raise
    except DeezerError as e:
            print("deezer error:", e)
            raise

tracks = get_top_tracks()

In [89]:
tracks.head(3)

,id,title,title_short,title_version,link,duration,rank,explicit_lyrics,explicit_content_lyrics,explicit_content_cover,preview,md5_image,position,artist,album,type,Genre
0,3602074142,Choosin' Texas,Choosin' Texas,,https://www.deezer.com/track/3602074142,232,978085,False,0,0,https://cdnt-preview.dzcdn.net/api/1/1/5/b/5/0...,749ad2ad7609ec768e342bec41078095,1,"{'id': 14867123, 'name': 'Ella Langley', 'link...","{'id': 837881302, 'title': 'Choosin' Texas', '...",track,Pop
1,3895241341,Boston,Boston,,https://www.deezer.com/track/3895241341,170,873262,False,0,0,https://cdnt-preview.dzcdn.net/api/1/1/a/d/f/0...,66d6baca9df5092af22a710ce10d39cf,2,"{'id': 181157207, 'name': 'STELLA LEFTY', 'lin...","{'id': 937796461, 'title': 'Boston', 'cover': ...",track,Pop
2,4250696681,Nicole Kidman,Nicole Kidman,,https://www.deezer.com/track/4250696681,181,990767,True,0,0,https://cdnt-preview.dzcdn.net/api/1/1/0/8/b/0...,68502d8730fa4ce6a41e2cb4db6d2fb7,3,"{'id': 306607671, 'name': 'ADÉLA', 'link': 'ht...","{'id': 1064827231, 'title': 'PRIMA', 'cover': ...",track,Pop


In [90]:
# expanding out the artist column from a dictionary to separate columns to be added back to the tracks dataframe
artist_info_df = pd.json_normalize(tracks['artist'])
artist_info_df = artist_info_df.rename(columns={'id':'artist_id','name':'artist_name'}, errors='ignore')[['artist_id','artist_name','link','picture','tracklist']]
artist_info_df.head(3)

,artist_id,artist_name,link,picture,tracklist
0,14867123,Ella Langley,https://www.deezer.com/artist/14867123,https://api.deezer.com/artist/14867123/image,https://api.deezer.com/artist/14867123/top?lim...
1,181157207,STELLA LEFTY,https://www.deezer.com/artist/181157207,https://api.deezer.com/artist/181157207/image,https://api.deezer.com/artist/181157207/top?li...
2,306607671,ADÉLA,https://www.deezer.com/artist/306607671,https://api.deezer.com/artist/306607671/image,https://api.deezer.com/artist/306607671/top?li...


In [ ]:
## changed, rewrite this to get the artists from top tracks and get their number of fans
# def get_artist_data() -> pd.DataFrame:
#     try:
#         result = []
#         for artist_id in artists.head()['id']:
#             response = session.get(f'{url}/artist/{artist_id}', timeout=15)
#             response.raise_for_status()
#             body = response.json() # the whole response
#             if "error" in body:
#                 raise DeezerError('error in artists data body')
#             # dataframe transformation
#             # display(body)
#             # print(type(body))
#             result.append(body)
#         return result
#         # return pd.concat(result, ignore_index=True)[['id','name','Genre']]
#     except requests.RequestException as e:
#             print("request failed:", e)
#             raise
#     except DeezerError as e:
#             print("deezer error:", e)
#             raise

# artists_data = get_artist_data()

# DATA VALIDATION

add checks for nulls, missing genres, artist names, etc

# UPLOAD TO S3

In [ ]:
upload_df = tracks


# add all this to main so that it's a python script and package